In [1]:
import cv2
import numpy as np
import time

from pynq import Overlay, MMIO, allocate
from pynq.lib.video import VideoMode
from pynq.lib import AxiGPIO

In [2]:
BITFILE = "AES_SYS.bit"
ol = Overlay(BITFILE, download=True)

print(ol.ip_dict.keys())

NUM_CHUNKS = 24
IMG_WIDTH  = 1280
IMG_HEIGHT = 720
BYTES_PER_PIXEL = 4

FULL_IMAGE_PIXELS = IMG_WIDTH * IMG_HEIGHT
FULL_IMAGE_BYTES  = FULL_IMAGE_PIXELS * BYTES_PER_PIXEL

assert FULL_IMAGE_PIXELS % NUM_CHUNKS == 0

PIXELS_PER_CHUNK = FULL_IMAGE_PIXELS // NUM_CHUNKS
CHUNK_BYTES = PIXELS_PER_CHUNK * BYTES_PER_PIXEL

TAG_SIZE = 16

VIDEO_IN  = "input.mp4"
VIDEO_ENC = "rfc8439_enc_video.mp4"
VIDEO_DEC = "rfc8439_dec_video.mp4"

MAX_FRAMES = 60   # 先測 60 frame，完整影片改成 None

print("CHUNK_BYTES =", CHUNK_BYTES)

dict_keys(['axi_gpio_0', 'axi_gpio_1', 'axi_gpio_2', 'axi_gpio_3', 'axi_gpio_5', 'axi_gpio_6', 'axi_gpio_8', 'axi_gpio_10', 'axi_intc_0', 'axi_vdma_0', 'axi_cdma_0', 'processing_system7_0'])
CHUNK_BYTES = 153600


In [3]:
KEY = bytes.fromhex(
    "000102030405060708090a0b0c0d0e0f"
    "101112131415161718191a1b1c1d1e1f"
)

BASE_NONCE = bytes.fromhex("000000090000004a00000000")
AAD = b""

def align16(x):
    return (x + 15) & ~15

AD_LEN = len(AAD)
SRC_AAD_BASE_BYTES = 44
MSG_OFFSET = align16(SRC_AAD_BASE_BYTES + AD_LEN)

SRC_ENC_BYTES = MSG_OFFSET + CHUNK_BYTES
SRC_DEC_BYTES = MSG_OFFSET + CHUNK_BYTES + TAG_SIZE
DST_ENC_BYTES = CHUNK_BYTES + TAG_SIZE
DST_DEC_BYTES = CHUNK_BYTES

print("AD_LEN        =", AD_LEN)
print("MSG_OFFSET    =", MSG_OFFSET)
print("SRC_ENC_BYTES =", SRC_ENC_BYTES)
print("SRC_DEC_BYTES =", SRC_DEC_BYTES)

AD_LEN        = 0
MSG_OFFSET    = 48
SRC_ENC_BYTES = 153648
SRC_DEC_BYTES = 153664


In [4]:
GPIO1_ADDR = ol.ip_dict["axi_gpio_1"]["phys_addr"]   # msg_length
GPIO2_ADDR = ol.ip_dict["axi_gpio_2"]["phys_addr"]   # ad_length
GPIO3_ADDR = ol.ip_dict["axi_gpio_3"]["phys_addr"]   # mac_error
GPIO5_ADDR = ol.ip_dict["axi_gpio_5"]["phys_addr"]   # start
GPIO6_ADDR = ol.ip_dict["axi_gpio_6"]["phys_addr"]   # mode_decrypt
GPIO8_ADDR = ol.ip_dict["axi_gpio_8"]["phys_addr"]   # done

GPIO1_RANGE = ol.ip_dict["axi_gpio_1"]["addr_range"]
GPIO2_RANGE = ol.ip_dict["axi_gpio_2"]["addr_range"]
GPIO3_RANGE = ol.ip_dict["axi_gpio_3"]["addr_range"]
GPIO5_RANGE = ol.ip_dict["axi_gpio_5"]["addr_range"]
GPIO6_RANGE = ol.ip_dict["axi_gpio_6"]["addr_range"]
GPIO8_RANGE = ol.ip_dict["axi_gpio_8"]["addr_range"]

CDMA_ADDR  = ol.ip_dict["axi_cdma_0"]["phys_addr"]
CDMA_RANGE = ol.ip_dict["axi_cdma_0"]["addr_range"]

MSG_LENGTH = MMIO(GPIO1_ADDR, GPIO1_RANGE)
AD_LENGTH  = MMIO(GPIO2_ADDR, GPIO2_RANGE)
MAC_ERROR  = MMIO(GPIO3_ADDR, GPIO3_RANGE)
START      = MMIO(GPIO5_ADDR, GPIO5_RANGE)
MODE_DEC   = MMIO(GPIO6_ADDR, GPIO6_RANGE)
DONE       = MMIO(GPIO8_ADDR, GPIO8_RANGE)
cdma       = MMIO(CDMA_ADDR, CDMA_RANGE)

BRAM0_ADDR = 0xC0000000
BRAM1_ADDR = 0xC2000000

In [ ]:
def cdma_reset(timeout=1.0):
    cdma.write(0x00, 0x4)

    t0 = time.time()
    while (cdma.read(0x00) & 0x4) != 0:
        if time.time() - t0 > timeout:
            raise TimeoutError(f"CDMA reset timeout, status=0x{cdma.read(0x04):08x}")

    cdma.write(0x04, 0x7000)


def cdma_transfer(src_addr, dst_addr, nbytes, timeout=5.0):
    cdma_reset()

    cdma.write(0x00, 0x1)
    cdma.write(0x18, int(src_addr))
    cdma.write(0x20, int(dst_addr))
    cdma.write(0x28, int(nbytes))

    t0 = time.time()

    while True:
        status = cdma.read(0x04)

        if status & 0x770:
            raise RuntimeError(f"CDMA error, status=0x{status:08x}")

        if status & 0x1000:
            break

        if time.time() - t0 > timeout:
            raise TimeoutError(f"CDMA timeout, status=0x{status:08x}")

    cdma.write(0x04, 0x1000)


def run_rfc8439(mode_decrypt, msg_len, ad_len, timeout=20.0):
    for attempt in range(2):
        START.write(0x0, 0)
        time.sleep(0.005)

        MODE_DEC.write(0x0, 1 if mode_decrypt else 0)
        MSG_LENGTH.write(0x0, int(msg_len))
        AD_LENGTH.write(0x0, int(ad_len))

        time.sleep(0.001)

        START.write(0x0, 1)
        time.sleep(0.002)

        t0 = time.time()

        cleared = True
        while (DONE.read(0x0) & 0x1) != 0:
            if time.time() - t0 > timeout:
                START.write(0x0, 0)
                cleared = False
                break

        if not cleared:
            print(f"Warning: DONE did not clear, retry {attempt + 1}/2")
            continue

        START.write(0x0, 0)

        while (DONE.read(0x0) & 0x1) == 0:
            if time.time() - t0 > timeout:
                raise TimeoutError(
                    f"RFC8439 timeout waiting done, "
                    f"DONE=0x{DONE.read(0x0):08x}, "
                    f"MAC_ERROR=0x{MAC_ERROR.read(0x0):08x}"
                )

        return MAC_ERROR.read(0x0) & 0x1

    raise TimeoutError(
        f"DONE did not clear after retry, "
        f"DONE=0x{DONE.read(0x0):08x}, "
        f"MAC_ERROR=0x{MAC_ERROR.read(0x0):08x}"
    )
    
def nonce_for_frame_chunk(frame_idx, chunk_idx):
    n = bytearray(BASE_NONCE)
    tail = int.from_bytes(n[8:12], "little")
    msg_id = frame_idx * NUM_CHUNKS + chunk_idx
    n[8:12] = ((tail + msg_id) & 0xffffffff).to_bytes(4, "little")
    return bytes(n)


def fill_src_header(buf, nonce):
    buf[:] = 0
    buf[0:32] = np.frombuffer(KEY, dtype=np.uint8)
    buf[32:44] = np.frombuffer(nonce, dtype=np.uint8)

    if AD_LEN > 0:
        buf[44:44 + AD_LEN] = np.frombuffer(AAD, dtype=np.uint8)


def frame_bgr_to_plain_bytes(frame_bgr):
    frame_bgr = cv2.resize(frame_bgr, (IMG_WIDTH, IMG_HEIGHT))
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    r = frame_rgb[:, :, 0].astype(np.uint32)
    g = frame_rgb[:, :, 1].astype(np.uint32)
    b = frame_rgb[:, :, 2].astype(np.uint32)

    words = ((r << 16) | (g << 8) | b).astype(np.uint32).reshape(-1)
    return np.ascontiguousarray(words).view(np.uint8)


def bytes_to_bgr_image(data_bytes):
    words = np.frombuffer(np.ascontiguousarray(data_bytes).tobytes(), dtype=np.uint32)
    words = words.reshape((IMG_HEIGHT, IMG_WIDTH))

    rr = ((words >> 16) & 0xFF).astype(np.uint8)
    gg = ((words >> 8)  & 0xFF).astype(np.uint8)
    bb = ( words        & 0xFF).astype(np.uint8)

    return cv2.merge([bb, gg, rr])

In [6]:
src_dma = allocate(shape=(SRC_DEC_BYTES,), dtype=np.uint8, cacheable=False)
dst_dma = allocate(shape=(DST_ENC_BYTES,), dtype=np.uint8, cacheable=False)

def process_video_frame(frame_bgr, frame_idx):
    plain_bytes = frame_bgr_to_plain_bytes(frame_bgr)

    enc_bytes = np.zeros((FULL_IMAGE_BYTES,), dtype=np.uint8)
    dec_bytes = np.zeros((FULL_IMAGE_BYTES,), dtype=np.uint8)
    tags = np.zeros((NUM_CHUNKS, TAG_SIZE), dtype=np.uint8)

    for chunk_idx in range(NUM_CHUNKS):
        start = chunk_idx * CHUNK_BYTES
        end = start + CHUNK_BYTES
        nonce = nonce_for_frame_chunk(frame_idx, chunk_idx)

        fill_src_header(src_dma, nonce)
        src_dma[MSG_OFFSET:MSG_OFFSET + CHUNK_BYTES] = plain_bytes[start:end]
        src_dma.flush()

        cdma_transfer(src_dma.physical_address, BRAM0_ADDR, SRC_ENC_BYTES)

        mac_err = run_rfc8439(
            mode_decrypt=False,
            msg_len=CHUNK_BYTES,
            ad_len=AD_LEN,
            timeout=20.0
        )

        if mac_err:
            print(f"Warning: encrypt mac_error frame {frame_idx}, chunk {chunk_idx}")

        cdma_transfer(BRAM1_ADDR, dst_dma.physical_address, DST_ENC_BYTES)
        dst_dma.invalidate()

        enc_bytes[start:end] = np.array(dst_dma[:CHUNK_BYTES], dtype=np.uint8)
        tags[chunk_idx, :] = np.array(
            dst_dma[CHUNK_BYTES:CHUNK_BYTES + TAG_SIZE],
            dtype=np.uint8
        )

    auth_fail = False

    for chunk_idx in range(NUM_CHUNKS):
        start = chunk_idx * CHUNK_BYTES
        end = start + CHUNK_BYTES
        nonce = nonce_for_frame_chunk(frame_idx, chunk_idx)

        fill_src_header(src_dma, nonce)
        src_dma[MSG_OFFSET:MSG_OFFSET + CHUNK_BYTES] = enc_bytes[start:end]
        src_dma[MSG_OFFSET + CHUNK_BYTES:MSG_OFFSET + CHUNK_BYTES + TAG_SIZE] = tags[chunk_idx]
        src_dma.flush()

        cdma_transfer(src_dma.physical_address, BRAM0_ADDR, SRC_DEC_BYTES)

        mac_err = run_rfc8439(
            mode_decrypt=True,
            msg_len=CHUNK_BYTES,
            ad_len=AD_LEN,
            timeout=20.0
        )

        if mac_err:
            auth_fail = True
            print(f"AUTH FAIL frame {frame_idx}, chunk {chunk_idx}")

        cdma_transfer(BRAM1_ADDR, dst_dma.physical_address, DST_DEC_BYTES)
        dst_dma.invalidate()

        dec_bytes[start:end] = np.array(dst_dma[:CHUNK_BYTES], dtype=np.uint8)

    enc_img_bgr = bytes_to_bgr_image(enc_bytes)
    dec_img_bgr = bytes_to_bgr_image(dec_bytes)

    roundtrip_ok = (not auth_fail) and np.array_equal(dec_bytes, plain_bytes)

    return enc_img_bgr, dec_img_bgr, roundtrip_ok

In [7]:
cap = cv2.VideoCapture(VIDEO_IN)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_IN}")

fps = cap.get(cv2.CAP_PROP_FPS)
print("input fps:", cap.get(cv2.CAP_PROP_FPS))
print("output fps:", fps)
if fps <= 0:
    fps = 10

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

enc_writer = cv2.VideoWriter(VIDEO_ENC, fourcc, fps, (IMG_WIDTH, IMG_HEIGHT))
dec_writer = cv2.VideoWriter(VIDEO_DEC, fourcc, fps, (IMG_WIDTH, IMG_HEIGHT))

frame_idx = 0
t_all = time.time()

while True:
    if MAX_FRAMES is not None and frame_idx >= MAX_FRAMES:
        break

    ret, frame_bgr = cap.read()
    if not ret:
        break

    t0 = time.time()

    enc_img_bgr, dec_img_bgr, ok = process_video_frame(frame_bgr, frame_idx)

    enc_writer.write(enc_img_bgr)
    dec_writer.write(dec_img_bgr)

    dt = time.time() - t0

    print(
        f"frame {frame_idx}: "
        f"{'PASS' if ok else 'FAIL'}, "
        f"time={dt:.3f}s, "
        f"fps={1.0 / dt:.2f}"
    )

    frame_idx += 1

cap.release()
enc_writer.release()
dec_writer.release()

print("Video encrypt/decrypt done")
print("frames:", frame_idx)
print("total time:", time.time() - t_all)
print("saved:", VIDEO_ENC)
print("saved:", VIDEO_DEC)

input fps: 30.0
output fps: 30.0
frame 0: PASS, time=2.031s, fps=0.49
frame 1: PASS, time=1.944s, fps=0.51
frame 2: PASS, time=2.040s, fps=0.49
frame 3: PASS, time=2.068s, fps=0.48
frame 4: PASS, time=2.083s, fps=0.48
frame 5: PASS, time=2.082s, fps=0.48
frame 6: PASS, time=2.072s, fps=0.48
frame 7: PASS, time=2.076s, fps=0.48
frame 8: PASS, time=2.079s, fps=0.48
frame 9: PASS, time=2.061s, fps=0.49
frame 10: PASS, time=2.049s, fps=0.49
frame 11: PASS, time=2.030s, fps=0.49
frame 12: PASS, time=1.954s, fps=0.51
frame 13: PASS, time=2.037s, fps=0.49
frame 14: PASS, time=2.031s, fps=0.49
frame 15: PASS, time=2.081s, fps=0.48
frame 16: PASS, time=2.034s, fps=0.49
frame 17: PASS, time=2.047s, fps=0.49
frame 18: PASS, time=2.041s, fps=0.49
frame 19: PASS, time=2.041s, fps=0.49
frame 20: PASS, time=2.042s, fps=0.49
frame 21: PASS, time=2.081s, fps=0.48
frame 22: PASS, time=2.048s, fps=0.49
frame 23: PASS, time=2.026s, fps=0.49
frame 24: PASS, time=1.972s, fps=0.51
frame 25: PASS, time=2.038s

In [ ]:
# =========================================================
# Stable HDMI / VDMA video display
# original 30fps -> displayed as 5fps by frame skipping
# encrypted/decrypted already 5fps
# sw = 0: original
# sw = 1: encrypted
# sw >=2: decrypted
# =========================================================
import time
import numpy as np
import cv2
from pynq import allocate
from pynq.lib.video import VideoMode
from pynq.lib import AxiGPIO

HDMI_X_SHIFT_PIXELS = -470
HDMI_BYTE_PHASE = 1
USE_BGR2RGB = True

def prepare_for_hdmi(frame_bgr):
    frame_bgr = cv2.resize(frame_bgr, (IMG_WIDTH, IMG_HEIGHT))

    if USE_BGR2RGB:
        frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    else:
        frame = frame_bgr

    frame = np.ascontiguousarray(frame, dtype=np.uint8)

    row_bytes = IMG_WIDTH * 3
    byte_shift = HDMI_X_SHIFT_PIXELS * 3 + HDMI_BYTE_PHASE

    rows = frame.reshape(IMG_HEIGHT, row_bytes)
    fixed_rows = np.roll(rows, byte_shift, axis=1)

    return np.ascontiguousarray(fixed_rows.reshape(IMG_HEIGHT, IMG_WIDTH, 3))


def read_looped(cap):
    ret, frame = cap.read()

    if ret:
        return frame

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    ret, frame = cap.read()

    if not ret:
        raise RuntimeError("Video playback read failed")

    return frame


def read_orig_5fps(cap, skip):
    frame = None

    for _ in range(skip):
        ret, f = cap.read()

        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, f = cap.read()

            if not ret:
                raise RuntimeError("Original video playback read failed")

        frame = f

    return frame


# ---------------------------------------------------------
# Stop old resources if they exist
# ---------------------------------------------------------
try:
    cap_orig.release()
except:
    pass

try:
    cap_enc.release()
except:
    pass

try:
    cap_dec.release()
except:
    pass

try:
    vdma.writechannel.stop()
    time.sleep(0.1)
except:
    pass


# ---------------------------------------------------------
# Open videos
# ---------------------------------------------------------
cap_orig = cv2.VideoCapture(VIDEO_IN)
cap_enc  = cv2.VideoCapture(VIDEO_ENC)
cap_dec  = cv2.VideoCapture(VIDEO_DEC)

if not cap_orig.isOpened():
    raise RuntimeError(f"Cannot open {VIDEO_IN}")
if not cap_enc.isOpened():
    raise RuntimeError(f"Cannot open {VIDEO_ENC}")
if not cap_dec.isOpened():
    raise RuntimeError(f"Cannot open {VIDEO_DEC}")

orig_fps = cap_orig.get(cv2.CAP_PROP_FPS)
enc_fps  = cap_enc.get(cv2.CAP_PROP_FPS)

if orig_fps <= 0:
    orig_fps = 30
if enc_fps <= 0:
    enc_fps = 5

fps_play = enc_fps
frame_period = 1.0 / fps_play
ORIG_SKIP = max(1, int(round(orig_fps / fps_play)))

orig_count = int(cap_orig.get(cv2.CAP_PROP_FRAME_COUNT))
enc_count  = int(cap_enc.get(cv2.CAP_PROP_FRAME_COUNT))
dec_count  = int(cap_dec.get(cv2.CAP_PROP_FRAME_COUNT))

print("HDMI playback")
print("orig_fps:", orig_fps)
print("enc_fps:", enc_fps)
print("fps_play:", fps_play)
print("ORIG_SKIP:", ORIG_SKIP)
print("orig_count:", orig_count)
print("enc_count:", enc_count)
print("dec_count:", dec_count)


# ---------------------------------------------------------
# Start VDMA
# ---------------------------------------------------------
vdma = ol.axi_vdma_0

try:
    vdma.writechannel.stop()
    time.sleep(0.1)
except:
    pass

mode = VideoMode(IMG_WIDTH, IMG_HEIGHT, 24)
vdma.writechannel.mode = mode
vdma.writechannel.start()
time.sleep(0.1)


# ---------------------------------------------------------
# Switch GPIO
# ---------------------------------------------------------
sw_1 = AxiGPIO(ol.ip_dict["axi_gpio_10"]).channel1


# ---------------------------------------------------------
# Double buffer
# ---------------------------------------------------------
display_buf = [
    allocate(shape=(IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.uint8, cacheable=False),
    allocate(shape=(IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.uint8, cacheable=False),
]

names = ["original", "encrypted", "decrypted", "decrypted"]

buf_idx = 0
last_sel = -1
next_t = time.perf_counter()


# ---------------------------------------------------------
# Show first frame immediately
# ---------------------------------------------------------
frame_orig = read_orig_5fps(cap_orig, ORIG_SKIP)
frame_enc  = read_looped(cap_enc)
frame_dec  = read_looped(cap_dec)

display_buf[buf_idx][:] = prepare_for_hdmi(frame_orig)
vdma.writechannel.writeframe(display_buf[buf_idx])
buf_idx ^= 1

print("show: original")


# ---------------------------------------------------------
# Playback loop
# ---------------------------------------------------------
while True:
    frame_orig = read_orig_5fps(cap_orig, ORIG_SKIP)
    frame_enc  = read_looped(cap_enc)
    frame_dec  = read_looped(cap_dec)

    sel = sw_1.read() & 0x3

    if sel == 0:
        chosen = frame_orig
    elif sel == 1:
        chosen = frame_enc
    else:
        chosen = frame_dec

    display_buf[buf_idx][:] = prepare_for_hdmi(chosen)
    vdma.writechannel.writeframe(display_buf[buf_idx])

    buf_idx ^= 1

    if sel != last_sel:
        print("show:", names[sel])
        last_sel = sel

    next_t += frame_period
    sleep_t = next_t - time.perf_counter()

    if sleep_t > 0:
        time.sleep(sleep_t)
    else:
        next_t = time.perf_counter()

show: original
show: encrypted
show: original
show: encrypted
show: original
show: decrypted
show: original


KeyboardInterrupt: 